# Data Preprocessing and Feature Engineering

Prepare the housing data using the saved train/test split and decisions documented during EDA. Imputation, encoding, and scaling will be fitted within training folds to prevent data leakage. The test set will remain reserved for final evaluation.

## 1. Import Required Libraries

In [37]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone
import joblib


## 2. Load the Saved Training Split

Reconstruct the training dataset using the previously saved row indices. Preserve the original row order of the raw dataset so that the saved indices identify the correct records.

In [2]:
project_root = Path.cwd().parent

df = pd.read_csv(project_root / "data/raw/melb_data.csv")
train_indices = pd.read_csv(
    project_root / "data/processed/train_indices.csv"
)["row_index"]

train_df = df.loc[train_indices].copy()

train_df.shape

(10864, 21)

### 3.1 Treat Zero Building Area as Missing

EDA identified 15 zero building-area values. Treat these as unavailable measurements while retaining the property records.

In [3]:
clean_train = train_df.copy()

clean_train["BuildingArea"] = clean_train["BuildingArea"].replace(0, np.nan)

clean_train["BuildingArea"].isna().sum()

np.int64(5144)

### 3.2 Treat Zero Land Size in Houses as Missing

Treat zero `Landsize` values for houses as unavailable information. Retain zeros for units and townhouses because shared-land recording conventions may apply.

In [4]:
mask = (clean_train["Type"] == "h") & (clean_train["Landsize"] == 0)

clean_train.loc[mask, "Landsize"] = np.nan

clean_train["Landsize"].isna().sum()

np.int64(138)

### 3.3 Treat Zero Bathroom Counts as Missing

Treat zero bathroom counts as unavailable information based on the EDA findings. Retain all property records.

In [5]:
clean_train["Bathroom"] = clean_train["Bathroom"].replace(0, np.nan)

clean_train["Bathroom"].isna().sum()

np.int64(25)

The 25 zero bathroom counts were treated as missing because they may represent unavailable information. All property records were retained; these missing values will be handled during preprocessing.

### 3.4 Treat Zero Bedroom Counts as Missing

Zero values in `Bedroom2` are treated as unavailable bedroom counts for this analysis. All records are retained, and the original `Rooms` values remain unchanged.

In [6]:
clean_train["Bedroom2"] = clean_train["Bedroom2"].replace(0, np.nan)

clean_train["Bedroom2"].isna().sum()

np.int64(12)

The 12 zero values in `Bedroom2` were treated as missing under the documented cleaning assumption. All records were retained, and no values were copied from `Rooms`.

### 3.5 Review the Extreme Bedroom Count

EDA identified a property with 3 rooms but 20 recorded bedrooms. Inspect this discrepancy before deciding how to handle it during preprocessing.

In [7]:
clean_train.loc[
    clean_train["Bedroom2"] == 20,
    ["Rooms", "Bedroom2", "Bathroom", "Type", "BuildingArea"]
]

,Rooms,Bedroom2,Bathroom,Type,BuildingArea
7404,3,20.0,1.0,h,NaN


The record contains 3 rooms, 20 bedrooms, and 1 bathroom, with building area unavailable. Given this substantial inconsistency, the bedroom count is treated as missing as a documented cleaning assumption. The property is retained without guessing a replacement value.

In [8]:
clean_train.loc[clean_train["Bedroom2"] == 20, "Bedroom2"] = np.nan

clean_train["Bedroom2"].isna().sum()

np.int64(13)

`Bedroom2` now contains 13 missing values: 12 zero counts and one inconsistent count of 20. All property records were retained.

### 3.6 Parse Sale Dates

Convert sale dates to datetime using the dataset's day/month/year format. These dates will support construction-year consistency checks and the calculation of property age at sale.

In [9]:
clean_train["Date"] = pd.to_datetime(
    clean_train["Date"], format="%d/%m/%Y", errors="coerce"
)

clean_train["Date"].isna().sum()

np.int64(0)

**Observation**

All 10,864 sale dates were successfully parsed using the day/month/year format. No missing dates remained after conversion.

### 3.7 Calculate Property Age at Sale

Calculate approximate property age by subtracting the construction year from the sale year. Missing construction years produce missing ages. Negative ages are identified for review.

In [10]:
clean_train["Age_At_Sale"] = (
    clean_train["Date"].dt.year - clean_train["YearBuilt"]
)

(clean_train["Age_At_Sale"] < 0).sum()

np.int64(6)

**Observation**

Six properties have a recorded construction year later than their sale year, producing negative ages. These may reflect recording errors or planned construction, but the available data cannot confirm the cause. Their calculated ages require handling before use in modeling.

### 3.8 Treat Negative Property Ages as Missing

Negative ages are treated as missing because they cannot represent completed property age at sale. All records and original construction years are retained without guessing replacement ages.

In [11]:
clean_train.loc[
    clean_train["Age_At_Sale"] < 0, "Age_At_Sale"
] = np.nan

clean_train["Age_At_Sale"].isna().sum()

np.int64(4292)

**Observation**

`Age_At_Sale` contains 4,292 missing values: 4,286 from missing construction years and 6 from negative calculated ages. All property records and original `YearBuilt` values were retained. Valid zero ages, representing construction and sale in the same year, remain unchanged.

### 3.9 Review Missing Values After Cleaning

Summarize missing values after applying the documented cleaning rules and creating `Age_At_Sale`. This check verifies the resulting counts before imputation.

In [12]:
missing_after_cleaning = clean_train.isna().sum()

missing_after_cleaning[missing_after_cleaning > 0]

Bedroom2          13
Bathroom          25
Car               50
Landsize         138
BuildingArea    5144
YearBuilt       4286
CouncilArea     1090
Age_At_Sale     4292
dtype: int64

**Observation**

Missing-value counts match the documented cleaning rules. `BuildingArea` has 5,144 missing values, while `Age_At_Sale` has 4,292, including six negative ages treated as missing. Additional missing values occur in `Bedroom2`, `Bathroom`, and house `Landsize` following cleaning. Existing missing values in `Car`, `YearBuilt`, and `CouncilArea` remain unchanged. Imputation has not yet been applied.

### 3.10 Verify Record and Target Preservation

Confirm that cleaning preserved the training records, their index order, and the original target values.

In [13]:
print("Shape after cleaning:", clean_train.shape)
print("Same row indices:", clean_train.index.equals(train_df.index))
print("Price unchanged:", clean_train["Price"].equals(train_df["Price"]))

Shape after cleaning: (10864, 22)
Same row indices: True
Price unchanged: True


**Observation**

The cleaned training data contains 10,864 records and 22 columns. Row indices and their order match the original training split, and all `Price` values remain unchanged. The additional column is the derived `Age_At_Sale` feature.

### 3.11 Save the Cleaned Training Data

Save the cleaned training data with its original row indices. Missing values are retained because imputation, encoding, and scaling will be fitted within training folds during modeling. The raw dataset remains unchanged.

In [14]:
clean_train.to_csv(
    project_root / "data/processed/train_cleaned.csv",
    index=True,
    index_label="row_index"
)

### 3.12 Verify the Saved Training Data

Reload the saved CSV to confirm its shape, original row indices, and retained missing values.

In [15]:
saved_train = pd.read_csv(
    project_root / "data/processed/train_cleaned.csv",
    index_col="row_index",
    parse_dates=["Date"]
)

print("Saved shape:", saved_train.shape)
print("Same row indices:", saved_train.index.equals(clean_train.index))
print("Same missing counts:",
      saved_train.isna().sum().equals(clean_train.isna().sum()))

Saved shape: (10864, 22)
Same row indices: True
Same missing counts: True


**Observation**

The saved training data was successfully reloaded with 10,864 records and 22 columns. Original row indices and per-column missing-value counts match the cleaned DataFrame, confirming that these were preserved during export.

## 4. Define Model Inputs

Define the prediction target and select input features using the EDA findings and their availability at prediction time.

### 4.1 Define the Target

Use the original sale price as the regression target. Keep it separate from the predictor features.

In [16]:
y_train = clean_train["Price"].copy()

y_train.shape

(10864,)

**Observation**

The target contains 10,864 price values, one for each training record. Original prices were retained without transformation and stored separately in `y_train`.

### 4.2 Define the Prediction Scenario

The initial model will estimate property prices before sale completion. Predictors should therefore be available at that point.

`Price` is reserved as the target. `Address` is excluded from the initial model because it is nearly unique. `Method` is excluded because its availability before sale is not established. The actual sale `Date` and derived `Age_At_Sale` are also excluded to avoid relying on a future sale date.

`Age_At_Sale` remains in the cleaned dataset for quality checks. These exclusions define the initial modeling setup rather than imply that the features have no predictive value.

### 4.3 Create the Candidate Predictor Dataset

Exclude the target and the columns identified in the prediction-scenario review. Retain the remaining columns as candidate predictors for further preprocessing decisions. The cleaned dataset remains unchanged.

In [17]:
excluded_columns = [
    "Price", "Address", "Method", "Date", "Age_At_Sale"
]

X_train = clean_train.drop(columns=excluded_columns).copy()

X_train.shape

(10864, 17)

**Observation**

The candidate predictor dataset contains 10,864 records and 17 features after excluding `Price`, `Address`, `Method`, `Date`, and `Age_At_Sale`. The cleaned dataset remains unchanged. Further feature decisions will be made before defining the preprocessing pipeline.

### 4.4 Reduce Overlap Between Room Count Features

Retain `Rooms` and exclude `Bedroom2` from the initial model. These features agree exactly in 95.07% of training records, while `Bedroom2` also contains documented inconsistencies.

This is an initial feature-set decision. The additional value of `Bedroom2` can later be assessed through training cross-validation.

In [18]:
X_train = X_train.drop(columns=["Bedroom2"])

X_train.shape

(10864, 16)

**Observation**

The initial predictor set contains 16 features after excluding `Bedroom2`. `Rooms` was retained to represent room-count information while reducing overlap between the two features. All 10,864 training records remain available.

### 4.5 Handle Flagged Construction Years in Model Inputs

Treat construction years recorded after the sale year as missing in the predictor dataset. These six records were identified during EDA, but their actual construction years remain unverified.

This is a documented modeling assumption. Original construction years remain preserved in the cleaned dataset.

In [19]:
future_year = clean_train["YearBuilt"] > clean_train["Date"].dt.year

X_train.loc[future_year, "YearBuilt"] = np.nan

X_train["YearBuilt"].isna().sum()

np.int64(4292)

**Observation**

`YearBuilt` now contains 4,292 missing values in the predictor dataset, including six construction years recorded after sale. These flagged values were treated as missing without guessing replacements. All training records and the original construction years in the cleaned dataset were retained.

### 4.6 Treat Postcode as a Categorical Feature

Postcodes identify geographic areas rather than numerical quantities. Convert them to text so they can be processed as categorical predictors.

In [20]:
X_train["Postcode"] = X_train["Postcode"].astype("int64").astype(str)

X_train["Postcode"].head()

12796    3149
9642     3149
3207     3122
1698     3054
761      3204
Name: Postcode, dtype: object

**Observation**

`Postcode` was converted from numeric values to text labels, preserving the area codes without decimal suffixes. It will be processed as a categorical feature rather than a numerical quantity.

### 4.7 Separate Numerical and Categorical Features

Group the selected predictors by data type to assign appropriate preprocessing. Numerical features will receive median imputation and scaling. Categorical features will receive missing-category imputation and one-hot encoding.

In [21]:
numerical_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(include="object").columns.tolist()

print("Numerical:", numerical_features)
print("Categorical:", categorical_features)

Numerical: ['Rooms', 'Distance', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'Lattitude', 'Longtitude', 'Propertycount']
Categorical: ['Suburb', 'Type', 'SellerG', 'Postcode', 'CouncilArea', 'Regionname']


**Observation**

The predictor set contains 10 numerical and 6 categorical features. `Postcode` is correctly grouped as categorical. These groups will receive separate preprocessing within the pipeline; no imputation, encoding, or scaling has been applied yet.

## 5. Build the Preprocessing Pipeline

Define preprocessing steps that will be fitted within each training fold during cross-validation.

### 5.1 Define Numerical Preprocessing

Fill missing numerical values using each feature's training median, then standardize the features. Defining this pipeline does not yet calculate medians or scaling statistics.

In [22]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

### 5.2 Define Categorical Preprocessing

Replace missing categorical values with an explicit "Unknown" label, then apply one-hot encoding. Categories will be learned from the training folds. Previously unseen categories will be handled without raising an error.

In [23]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant", fill_value="Unknown"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

### 5.3 Combine Numerical and Categorical Preprocessing

Use a ColumnTransformer to apply each preprocessing pipeline to its assigned columns and combine their outputs. No preprocessing statistics or categories are learned at this stage.

In [24]:
preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [25]:
preprocessor

,transformers,"[('numerical', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


### 5.4 Verify Preprocessing Feature Coverage

Confirm that every predictor is assigned to a preprocessing group and that the numerical and categorical feature lists do not overlap.

In [26]:
assigned_features = numerical_features + categorical_features

print("All features covered:",
      set(assigned_features) == set(X_train.columns))

print("No repeated assignments:",
      len(assigned_features) == len(set(assigned_features)))

All features covered: True
No repeated assignments: True


**Observation**

All 16 predictors are assigned to the preprocessing pipelines: 10 numerical and 6 categorical features. No columns are omitted or assigned more than once. The preprocessor remains unfitted; no statistics or categories have been learned yet.

### 5.5 Run a Small Preprocessing Check

Fit a separate copy of the preprocessor on the first 100 training records to check that the transformations execute together. This is an implementation check, not model evaluation. The transformed sample will not be used for cross-validation.

In [32]:
preview_preprocessor = clone(preprocessor)
preview_data = preview_preprocessor.fit_transform(X_train.head(100))

print("Before preprocessing:", X_train.head(100).shape)
print("After preprocessing:", preview_data.shape)

Before preprocessing: (100, 16)
After preprocessing: (100, 217)


**Observation**

The preprocessing trial transformed 100 records with 16 input features into 217 output features: 10 numerical and 207 one-hot encoded features. All sample rows were retained. The transformations executed successfully using a separate preprocessor copy; the original preprocessor remains unfitted.

### 5.6 Check the Transformed Sample for Non-Finite Values

Verify that the transformed sample contains no missing or infinite values. This check applies only to the 100-record preprocessing trial.

In [33]:
preview_array = (
    preview_data.toarray()
    if hasattr(preview_data, "toarray")
    else preview_data
)

print("All values finite:", np.isfinite(preview_array).all())

All values finite: True


**Observation**

All values in the transformed 100-record sample are finite, with no missing or infinite values remaining. This confirms that the preprocessing trial produced valid numerical inputs for this sample. The original preprocessor remains unfitted for later use within cross-validation.

## 6. Save Modeling Inputs

Save the selected predictors and target separately for reuse in modeling notebooks. Predictors retain missing values and remain unscaled and unencoded so that learned preprocessing can be fitted within cross-validation.

### 6.1 Save Training Predictors and Target

In [34]:
X_train.to_csv(
    project_root / "data/processed/X_train.csv",
    index_label="row_index"
)

y_train.to_csv(
    project_root / "data/processed/y_train.csv",
    index_label="row_index"
)

### 6.2 Save the Unfitted Preprocessor

Save the preprocessing configuration, including its feature assignments, for reuse in modeling notebooks. The preprocessor remains unfitted and must be fitted together with each model within cross-validation.

In [36]:
(project_root / "models").mkdir(exist_ok=True)

joblib.dump(
    preprocessor,
    project_root / "models/preprocessor_unfitted.joblib"
)

['C:\\Users\\mamona\\Desktop\\house-price-linear-models\\models\\preprocessor_unfitted.joblib']

**Observation**

The unfitted preprocessor was saved as `models/preprocessor_unfitted.joblib`. It preserves the preprocessing steps and feature assignments without learned statistics or categories. During modeling, it will be fitted within each cross-validation training fold.

### 6.3 Verify Saved Predictor and Target Alignment

Reload the saved training inputs, preserving postcode labels as text. Confirm that predictors and target retain matching row indices and order.

In [38]:
saved_X = pd.read_csv(
    project_root / "data/processed/X_train.csv",
    index_col="row_index",
    dtype={"Postcode": str}
)

saved_y = pd.read_csv(
    project_root / "data/processed/y_train.csv",
    index_col="row_index"
)["Price"]

print("Predictor shape:", saved_X.shape)
print("Target shape:", saved_y.shape)
print("Rows aligned:", saved_X.index.equals(saved_y.index))

Predictor shape: (10864, 16)
Target shape: (10864,)
Rows aligned: True


**Observation**

The saved predictors contain 10,864 records and 16 features, while the target contains 10,864 price values. Their row indices and order match, confirming alignment after reloading. `Postcode` was explicitly loaded as text to preserve its categorical interpretation.

## 7. Prepare the Held-Out Test Inputs

Apply the established cleaning and feature-selection rules to the original saved test split. Do not fit preprocessing on the test data or use it to revise modeling decisions.

### 7.1 Locate the Saved Split Indices

In [39]:
[
    file.name
    for file in (project_root / "data/processed").glob("*indices*.csv")
]

['test_indices.csv', 'train_indices.csv']

### 7.2 Load the Saved Test Split

Reconstruct the held-out test dataset using its saved row indices. Preserve the original split without reshuffling or selecting a new test sample.

In [40]:
test_indices = pd.read_csv(
    project_root / "data/processed/test_indices.csv"
)["row_index"]

test_df = df.loc[test_indices].copy()

test_df.shape

(2716, 21)

**Observation**

The held-out test dataset was reconstructed from the saved indices with 2,716 records and 21 original columns. No new split was created, and no cleaning or learned preprocessing has yet been applied to these records.

### 7.3 Verify Train–Test Index Separation

Confirm that the reconstructed training and test datasets have no overlapping row indices. This verifies row-level separation of the saved split.

In [41]:
overlap = train_df.index.intersection(test_df.index)

print("Overlapping row indices:", len(overlap))

Overlapping row indices: 0


**Observation**

No overlapping row indices were found between the 10,864 training records and 2,716 test records. The saved split preserves row-level separation; this check does not establish separation by property address or sale date.

### 7.4 Treat Zero Test Building Areas as Missing

Apply the established training-data rule by replacing zero building areas with missing values. Retain all test records without estimating replacement values from the test data.

In [42]:
clean_test = test_df.copy()

clean_test["BuildingArea"] = clean_test["BuildingArea"].replace(0, np.nan)

print("Zero building areas remaining:",
      clean_test["BuildingArea"].eq(0).sum())

Zero building areas remaining: 0


**Observation**

No zero building-area values remain after applying the established cleaning rule to the test data. All test records were retained, and no imputation was performed.

### 7.5 Treat Zero House Land Sizes as Missing

Apply the established rule by replacing zero land sizes for houses with missing values. Retain zero land sizes for units and townhouses, consistent with the training-data treatment.

In [43]:
mask = (clean_test["Type"] == "h") & (clean_test["Landsize"] == 0)

clean_test.loc[mask, "Landsize"] = np.nan

print("Zero house land sizes remaining:",
      ((clean_test["Type"] == "h") & (clean_test["Landsize"] == 0)).sum())

Zero house land sizes remaining: 0


**Observation**

No zero land sizes remain for houses in the test data after applying the established cleaning rule. Land-size values for units and townhouses were left unchanged, and all test records were retained.

### 7.6 Treat Zero Test Bathroom Counts as Missing

Replace zero bathroom counts with missing values, following the documented training-data cleaning assumption. Retain all test records without performing imputation.

In [44]:
clean_test["Bathroom"] = clean_test["Bathroom"].replace(0, np.nan)

print("Zero bathroom counts remaining:",
      clean_test["Bathroom"].eq(0).sum())

Zero bathroom counts remaining: 0


**Observation**

No zero bathroom counts remain after applying the established cleaning rule to the test data. All test records were retained, and missing values remain available for imputation using training-fitted preprocessing.

### 7.7 Apply the Established Bedroom-Count Rules

Replace bedroom counts of 0 and 20 with missing values, following the documented training-data assumptions. Retain all records. Although `Bedroom2` is excluded from the initial model, its treatment remains consistent across the cleaned datasets.

In [45]:
clean_test["Bedroom2"] = clean_test["Bedroom2"].replace([0, 20], np.nan)

print("Flagged bedroom counts remaining:",
      clean_test["Bedroom2"].isin([0, 20]).sum())

Flagged bedroom counts remaining: 0


**Observation**

No bedroom counts of 0 or 20 remain after applying the established cleaning rules to the test data. All records were retained. `Bedroom2` remains excluded from the initial model inputs.

### 7.8 Parse Test Sale Dates

Convert test sale dates using the established day/month/year format. These dates support the same construction-year consistency checks used for training and remain excluded from model inputs.

In [46]:
clean_test["Date"] = pd.to_datetime(
    clean_test["Date"], format="%d/%m/%Y", errors="coerce"
)

print("Missing dates after parsing:", clean_test["Date"].isna().sum())

Missing dates after parsing: 0


**Observation**

All 2,716 test sale dates were successfully parsed using the day/month/year format. No missing dates remained after conversion. Sale dates are retained for consistency checks and excluded from model inputs.

### 7.9 Calculate and Clean Test Property Ages

Calculate approximate age at sale using the same rule applied to training data. Treat negative ages as missing while preserving valid zero ages and original construction years. This derived feature remains excluded from the initial model.

In [47]:
clean_test["Age_At_Sale"] = (
    clean_test["Date"].dt.year - clean_test["YearBuilt"]
)

clean_test.loc[
    clean_test["Age_At_Sale"] < 0, "Age_At_Sale"
] = np.nan

print("Negative ages remaining:", clean_test["Age_At_Sale"].lt(0).sum())

Negative ages remaining: 0


**Observation**

No negative ages remain after applying the established age-cleaning rule to the test data. Valid zero ages and original construction years were preserved. `Age_At_Sale` remains excluded from the initial model inputs.

### 7.10 Select Test Predictors and Target

Select the same predictor columns in the same order as the training inputs. Store the original test prices separately for final evaluation.

In [48]:
X_test = clean_test[X_train.columns].copy()
y_test = clean_test["Price"].copy()

print("Test predictors:", X_test.shape)
print("Test target:", y_test.shape)

Test predictors: (2716, 16)
Test target: (2716,)


**Observation**

The test predictor dataset contains 2,716 records and the same 16 columns, in the same order, as the training inputs. The target contains 2,716 original price values stored separately for final evaluation. Construction-year handling and postcode conversion remain to be applied to the test predictors.

### 7.11 Handle Flagged Test Construction Years

Apply the established modeling assumption: treat construction years recorded after sale as missing in the test predictors. Preserve original construction years in the cleaned test dataset.

In [49]:
future_year_test = (
    clean_test["YearBuilt"] > clean_test["Date"].dt.year
)

X_test.loc[future_year_test, "YearBuilt"] = np.nan

print("Future construction years remaining:",
      (X_test["YearBuilt"] > clean_test["Date"].dt.year).sum())

Future construction years remaining: 0


**Observation**

No construction years later than the sale year remain in the test predictors after applying the established rule. Original construction years were preserved in the cleaned test dataset, and no replacement years were estimated.

### 7.12 Convert Test Postcodes to Categorical Labels

Convert test postcodes to text labels using the same format as the training predictors. This ensures consistent categorical representation across both datasets.

In [50]:
X_test["Postcode"] = X_test["Postcode"].astype("int64").astype(str)

print("Postcode dtype:", X_test["Postcode"].dtype)

Postcode dtype: object


**Observation**

Test postcodes were converted to text labels without decimal suffixes, matching the categorical representation used in training. No encoding was fitted on the test data.

### 7.13 Verify Test Input Consistency

Verify matching predictor columns, column order, and data types across training and test inputs. Confirm that test row indices and original target values were preserved.

In [52]:
print("Same columns and order:", X_test.columns.equals(X_train.columns))
print("Same data types:", X_test.dtypes.equals(X_train.dtypes))
print("Test rows preserved:", X_test.index.equals(test_df.index))
print("Target aligned:", X_test.index.equals(y_test.index))
print("Price unchanged:", y_test.equals(test_df["Price"]))

Same columns and order: True
Same data types: True
Test rows preserved: True
Target aligned: True
Price unchanged: True


**Observation**

Test predictors match the training inputs in column names, order, and data types. All original test row indices were preserved, and target values remain unchanged and correctly aligned. No imputation, encoding, or scaling was fitted on the test data.

### 7.14 Save the Prepared Test Data

Save the cleaned test dataset, selected predictors, and original target separately, preserving row indices. Missing values remain unfilled, and predictors remain unscaled and unencoded. Learned preprocessing will be applied using the training-fitted pipeline during final evaluation.

In [ ]:
clean_test.to_csv(
    project_root / "data/processed/test_cleaned.csv",
    index_label="row_index"
)

X_test.to_csv(
    project_root / "data/processed/X_test.csv",
    index_label="row_index"
)

y_test.to_csv(
    project_root / "data/processed/y_test.csv",
    index_label="row_index"
)

## 8. Preprocessing Summary and Modeling Handoff

The original split was preserved with 10,864 training records and 2,716 test records. Documented cleaning assumptions were applied consistently, while original target prices and row indices were retained.

The initial model uses 16 predictors: 10 numerical and 6 categorical features. `Address`, `Method`, `Date`, `Age_At_Sale`, and `Bedroom2` are excluded from model inputs; `Price` is stored separately as the target.

Numerical preprocessing uses median imputation followed by standardization. Categorical preprocessing uses an explicit "Unknown" label for missing values followed by one-hot encoding. A separate 100-record trial produced finite numerical outputs; this was an implementation check, not model evaluation.

Cleaned datasets, selected predictors, targets, and the unfitted preprocessor were saved. Imputation statistics, scaling statistics, and encoding categories must be learned within each cross-validation training fold by combining preprocessing and the estimator in one pipeline.

Unverified extreme measurements remain a limitation. The effects of alternative feature treatments should be assessed through training cross-validation. The held-out test set remains reserved for final evaluation.